# 03 - Model Training & Tuning

End-to-End Pipeline: Daten laden → Optuna Tuning → Training → Modell in DagsHub registrieren.

Alles wird auf **DagsHub MLflow** geloggt:
- Jeder Optuna-Trial als nested Run
- Bestes Modell wird in der Model Registry registriert
- Altes Modell wird nur ersetzt wenn das neue besser ist

In [ ]:
from backend.core.config import settings
print(f"DagsHub Repo:  {settings.dagshub.repo_full}")
print(f"Token gesetzt: {bool(settings.dagshub.token.get_secret_value())}")
print(f"Tracking URI:  {settings.dagshub.tracking_uri}")
print(f"Model Name:    {settings.mlflow.model_name}")
print(f"Data Dir:      {settings.database.data_dir}")

## 1. DagsHub + MLflow initialisieren

In [ ]:
import mlflow
from backend.infra.dagshub_init import init_dagshub

init_dagshub()
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment:          {settings.mlflow.experiment_name}")

## 2. Features laden

In [ ]:
from backend.infra.database import get_data_store

store = get_data_store()

try:
    df = store.load_features()
    print(f"Features geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
    print(f"Spalten: {list(df.columns[:10])}... (+{len(df.columns)-10} weitere)")
    print(f"Target-Verteilung:\n{df['Target'].value_counts()}")
except FileNotFoundError:
    print("⚠️  Keine Features vorhanden! Erst fetch_data ausführen:")
    print("   from backend.workflows.fetch_data import fetch_and_store")
    print("   fetch_and_store()")

## 2b. (Optional) Daten erst fetchen, falls noch nicht vorhanden

In [ ]:
# Nur ausführen wenn Cell 5 "FileNotFoundError" zeigt!
from backend.workflows.fetch_data import fetch_and_store
df_fetched, manifest = fetch_and_store()
print(f"Daten geladen: {manifest.row_count} rows, {len(manifest.columns)} cols")

# Jetzt Features laden
df = store.load_features()
print(f"Features: {df.shape}")

## 3. Train/Test Split & Scaling

In [ ]:
from backend.ml.training.extra_tree import ExtraTreesTrainer

trainer = ExtraTreesTrainer()
x_train, x_test, y_train, y_test = trainer.prepare(df)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Train Target: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
print(f"Test  Target: 0={sum(y_test==0)},  1={sum(y_test==1)}")

## 4. Optuna Hyperparameter-Tuning (10 Trials → DagsHub MLflow)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

with mlflow.start_run(run_name="train_ExtraTreesClassifier") as run:
    # --- Optuna Tuning ---
    from backend.ml.training.tuning.optuna_tuner import tune

    N_TRIALS = 10
    print(f"Starting Optuna tuning with {N_TRIALS} trials...")
    best_params = tune(trainer, x_train, y_train, x_test, y_test, n_trials=N_TRIALS)
    
    print(f"\nBeste Parameter:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    
    # --- Finales Model mit besten Params ---
    print("\nTraining final model with best params...")
    model = trainer.build(best_params)
    model.fit(x_train, y_train)
    
    preds = model.predict(x_test)
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds, average="weighted"),
    }
    print(f"\nErgebnis:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  F1:       {metrics['f1']:.4f}")
    
    # --- In DagsHub Model Registry registrieren ---
    from backend.ml.registry.model_registry import log_and_register
    
    model_uri = log_and_register(
        model=model,
        x_train=x_train,
        y_train=y_train,
        params=best_params,
        metrics=metrics,
    )
    
    print(f"\nModel URI: {model_uri}")
    print(f"Run ID:    {run.info.run_id}")
    print(f"\nSchau auf DagsHub: {settings.dagshub.tracking_uri}")

## 5. Modell verifizieren — aus DagsHub laden & testen

In [ ]:
from backend.core.prediction.mlflow_predict import MLflowPredictor

predictor = MLflowPredictor()
predictor.load_model()

# Test auf letzter Zeile
last_row = df.drop("Target", axis=1).tail(1)
pred = predictor.predict(last_row)[0]
proba = predictor.predict_proba(last_row)[0]

direction = "UP ↑" if pred == 1 else "DOWN ↓"
confidence = max(proba)
print(f"Prediction: {direction}")
print(f"Confidence: {confidence:.2%}")
print(f"Horizon:    {settings.stock.prediction_horizon_days} Tage")